# Sentiment Feature Generation 
Loads `Processed_Reviews.csv`, predicts sentiment for combined `Title + Text` using `cardiffnlp/twitter-roberta-base-sentiment`, and appends only `Sentiment` as the final column.

In [ ]:
import os
import pandas as pd
import torch
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from tqdm.auto import tqdm

# Optional HF dataset iterator to improve GPU throughput in pipeline
try:
    from datasets import Dataset
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
    from datasets import Dataset

# Config
INPUT_FILE = "Processed_Reviews.csv"
BATCH_SIZE = 32
MAX_TEXT_LEN = 256
PIPELINE_DEVICE = 0 if torch.cuda.is_available() else -1

def build_output_path(input_file):
    input_abs = os.path.abspath(input_file)
    input_dir = os.path.dirname(input_abs)
    base_name = os.path.splitext(os.path.basename(input_abs))[0]
    out_name = f"{base_name}_with_Sentiment.csv"

    # Save alongside input when writable; otherwise fall back to Kaggle working directory
    if os.path.isdir(input_dir) and os.access(input_dir, os.W_OK):
        return os.path.join(input_dir, out_name)

    kaggle_working = "/kaggle/working"
    os.makedirs(kaggle_working, exist_ok=True)
    return os.path.join(kaggle_working, out_name)

OUTPUT_FILE = build_output_path(INPUT_FILE)

# Load dataset
df = pd.read_csv(INPUT_FILE)

# Build combined text from Title + Text
title = df["Title"].fillna("").astype(str) if "Title" in df.columns else pd.Series([""] * len(df))
text = df["Text"].fillna("").astype(str) if "Text" in df.columns else pd.Series([""] * len(df))
combined_text = (title + " " + text).str.strip()

# Load sentiment model
sentiment_pipe = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=PIPELINE_DEVICE,
 )

# Prepare rows with actual text only
valid_indices = [i for i, t in enumerate(combined_text.tolist()) if isinstance(t, str) and t.strip()]
valid_texts = [combined_text.iloc[i] for i in valid_indices]

label_map = {"LABEL_0": "NEGATIVE", "LABEL_1": "NEUTRAL", "LABEL_2": "POSITIVE"}
predicted = [None] * len(combined_text)

if valid_texts:
    # Use Dataset + KeyDataset to stream batched data efficiently on GPU
    infer_ds = Dataset.from_dict({"text": valid_texts})
    outputs = sentiment_pipe(
        KeyDataset(infer_ds, "text"),
        truncation=True,
        max_length=MAX_TEXT_LEN,
        batch_size=BATCH_SIZE
    )

    for idx, out in zip(valid_indices, outputs):
        raw_label = str(out.get("label", "")).strip().upper()
        predicted[idx] = label_map.get(raw_label, raw_label if raw_label else None)

# Append only this new column at the end
df["Sentiment"] = predicted

# Save
df.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)
print("Final shape:", df.shape)
print(df[["Sentiment"]].head())